In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
# ============================================
# Device
# ============================================

dev = qml.device("default.qubit", wires=2)

In [3]:
# ============================================
# Hamiltonian
# H = -1/2 I + 3/2 Z0 - 2 Z1 + Z0 Z1
# ============================================

coeffs = [-0.5, 1.5, -2.0, 1.0]

observables = [
    qml.Identity(0),
    qml.PauliZ(0),
    qml.PauliZ(1),
    qml.PauliZ(0) @ qml.PauliZ(1)
]

H = qml.Hamiltonian(coeffs, observables)

# ============================================
# Variational Ansatz
# ============================================

def ansatz(params):

    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)

    qml.CNOT(wires=[0, 1])

# ============================================
# Quantum Node
# ============================================

@qml.qnode(dev)
def circuit(params):
    
    ansatz(params)

    return qml.expval(H)

In [4]:
# ============================================
# Optimization
# ============================================

params = np.random.uniform(0, 2*np.pi, 2, requires_grad=True)

optimizer = qml.GradientDescentOptimizer(stepsize=0.1)

steps = 100

for i in range(steps):

    params = optimizer.step(circuit, params)

    if (i + 1) % 10 == 0:
        energy = circuit(params)
        print(f"Step {i+1:3d} | Energy = {energy:.6f}")

Step  10 | Energy = -4.985010
Step  20 | Energy = -4.999994
Step  30 | Energy = -5.000000
Step  40 | Energy = -5.000000
Step  50 | Energy = -5.000000
Step  60 | Energy = -5.000000
Step  70 | Energy = -5.000000
Step  80 | Energy = -5.000000
Step  90 | Energy = -5.000000
Step 100 | Energy = -5.000000


In [5]:
# ============================================
# Final Results
# ============================================

final_energy = circuit(params)

print("\nOptimized Parameters:")
print(params)

print("\nGround-State Energy:")
print(final_energy)


Optimized Parameters:
[3.14159265 3.14159265]

Ground-State Energy:
-5.0


In [6]:
# ============================================
# Measure Most Probable State
# ============================================

@qml.qnode(dev)
def probabilities(params):

    ansatz(params)

    return qml.probs(wires=[0,1])

probs = probabilities(params)

print("\nProbabilities:")
print(probs)

states = ["00", "01", "10", "11"]

max_index = np.argmax(probs)

print(f"\nMost Probable Solution: {states[max_index]}")


Probabilities:
[2.07520401e-63 8.02458275e-32 1.00000000e+00 2.58605846e-32]

Most Probable Solution: 10


In [7]:
from platform import python_version

print('Pennylane Version:', qml.__version__)

print('Python Version:', python_version())
print('UdeA-Medellin (2026) ©')

Pennylane Version: 0.43.1
Python Version: 3.11.7
UdeA-Medellin (2026) ©
